In [1]:
import os
import pandas as pd
from pathlib import Path
from maomao.parsing.parsing_utils import *
from maomao.utils.constants import *

#### Processing and standardizing peptide datasets (ConsAMPHemo)

This notebook curates the **ConsAMPHemo** dataset by integrating multiple predefined splits (S1, S2, and S3) into a single standardized table. The pipeline normalizes peptide sequences, performs duplicate consistency checks, builds metadata, and exports the final curated dataset for downstream analysis.

- **Toxic effect / endpoint:** hemolytic
- **Source:** ConsAMPHemo
- **Sequence scope:** only non-modified peptide sequences are retained for the final dataset.

The pipeline performs the following steps:

- **Loads raw CSV files** from the `S1`, `S2`, and `S3` folders and concatenates them into a single DataFrame.
- **Standardizes column names and formats**:
  - `text` → `sequence`
  - `labels` → `label`
  - removes whitespace from peptide sequences.
- **Checks duplicated sequences** across all splits:
  - unique sequences are retained,
  - duplicates with consistent labels are collapsed,
  - sequences with conflicting labels are flagged as errors.
- **Builds metadata** from the project-wide Excel description file and appends QC statistics.
- **Exports curated outputs**:
  - `processed_hemolytic_dataset.csv` (merged and deduplicated dataset),
  - `detected_error_sequences.csv` (conflicting-label duplicates),
  - `metadata.json`.

In [2]:
name_source = "ConsAMPHemo"
name_task = "toxic_effect_classification"

# PATH_INPUT and PATH_EXPORT are imported from maomao.utils.constants
# Update them in constants.py according to the required input and export paths.

- Reading raw data

In [3]:
folders = ["S1", "S2", "S3"]
dfs = []

for folder in folders:
    for file in (Path(PATH_INPUT) / name_source / folder).glob("*.csv"):
        df = pd.read_csv(file)
        dfs.append(df)
 
df_ConsAMPHemo = pd.concat(dfs, ignore_index=True)

In [4]:
df_ConsAMPHemo = (
    df_ConsAMPHemo
    .rename(columns={"text": "sequence", "labels": "label"})
    .assign(
        sequence=lambda d: d["sequence"].astype(str).str.replace(" ", "", regex=False)
    )
    [["sequence", "label"]]
)
df_ConsAMPHemo.shape

(16370, 2)

- Checking duplicates

In [5]:
df_remove_duplicated, df_errors, df_unique = processing_duplicated(df_ConsAMPHemo, group_seq="sequence", sort_key="label")
df_full = pd.concat([df_unique, df_remove_duplicated], axis=0)

In [6]:
df_errors.shape

(23, 1)

- Working with metada

In [7]:
df_metada = read_metadata("../../raw_data/raw_data_description.xlsx", name_source)
dict_metadata = create_metada_with_multiple_values(df_metada)

In [8]:
dict_metadata.update({
    "number_of_raw_sequences": int(len(df_ConsAMPHemo)),
    "number_of_sequences_retained": len(df_full),
    "number_of_positive_sequences": int((df_full["label"] == 1).sum()),
    "number_of_negative_sequences": int((df_full["label"] == 0).sum()),
    "number_of_erroneous_sequences": int(len(df_errors)),
    "modified_sequences_included": False,
})

dict_metadata

{'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'No information',
 'year of publication': 2025,
 'last update date': datetime.datetime(2024, 9, 28, 0, 0),
 'download date': Timestamp('2025-10-17 00:00:00'),
 'file format': 'xlsx;csv',
 'peptide property': 'hemolytic, toxic',
 'dataset information': 'Concentration constants;Positive, Negative',
 'unit of measurement': 'No information',
 'obtaining negative dataset': 'Previously published model dataset, Sampling from another DB',
 'repository or server': 'https://github.com/Cpillar/ConsAMPHemo/tree/main/Dataset',
 'publication': 'https://onlinelibrary.wiley.com/doi/abs/10.1002/pro.70087',
 'number_of_raw_sequences': 16370,
 'number_of_sequences_retained': 5841,
 'number_of_positive_sequences': 2246,
 'number_of_negative_sequences': 3595,
 'number_of_erroneous_sequences': 23,
 'modified_sequences_included': False}

- Exporting data

In [9]:
os.makedirs(f"{PATH_EXPORT}/{name_task}/{name_source}/", exist_ok=True)
export_json(f"{PATH_EXPORT}/{name_task}/{name_source}/metadata.json", dict_metadata)

In [10]:
df_full.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_hemolytic_dataset.csv", index=False)
df_errors.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/detected_error_sequences.csv", index=False)